# ALLdis正式10,000次联合PSA运行 v1.3

本Notebook是正式运行入口。执行“全部运行”将先检查GBD接入、确定性模型和2×100次烟雾测试，随后运行20批×500次PSA，并自动合并和审计结果。不要同时打开多个正式运行进程。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'input' / 'psa').exists() else HERE.parent
print('Bundle root:', ROOT)

## 1. 正式运行前自动检查

In [ ]:
gbd_qc = pd.read_csv(ROOT / 'audit_gbd_update' / '00_GBD更新QC.csv')
det_qc = pd.read_csv(ROOT / 'results' / '00_QC.csv')
smoke_qc = pd.read_csv(ROOT / 'psa_smoke_recheck' / 'merged' / '00_QC.csv')
smoke_manifest = json.loads((ROOT / 'psa_smoke_recheck' / 'merged' / 'MERGED_MANIFEST.json').read_text(encoding='utf-8'))
psa_manifest = json.loads((ROOT / 'input' / 'psa' / 'PSA_INPUT_MANIFEST_v1.2.json').read_text(encoding='utf-8'))

assert (gbd_qc['status'] == 'PASS').all(), 'GBD update QC failed'
assert det_qc['passed'].all(), 'Deterministic QC failed'
assert (smoke_qc['status'] == 'PASS').all(), 'Smoke-test QC failed'
assert smoke_manifest['draws'] == 200 and not smoke_manifest['formal_psa_run']
assert psa_manifest['gbd_status'] == 'GBD_2010_2023_FINAL_HISTORY_LOCKED'
print(f"Preflight passed: GBD {len(gbd_qc)}/{len(gbd_qc)}, deterministic {len(det_qc)}/{len(det_qc)}, smoke {len(smoke_qc)}/{len(smoke_qc)}")

## 2. 运行20批×500次PSA

In [ ]:
RUN_FORMAL = True
WORKERS = 2

if not RUN_FORMAL:
    raise RuntimeError('RUN_FORMAL must remain True in the dedicated formal-run notebook.')
run_cmd = [sys.executable, str(ROOT / 'src' / 'run_psa_plan_v1_2.py'), '--run-type', 'FORMAL', '--workers', str(WORKERS), '--confirm-final-lock']
subprocess.run(run_cmd, cwd=ROOT, check=True)
print('All 20 formal batches completed.')

## 3. 合并正式结果并运行自动QC

In [ ]:
merge_cmd = [sys.executable, str(ROOT / 'src' / 'merge_and_qc_psa_v1_2.py'), '--input-root', str(ROOT / 'psa_formal'), '--output-dir', str(ROOT / 'psa_formal' / 'merged'), '--formal']
subprocess.run(merge_cmd, cwd=ROOT, check=True)
formal_manifest = json.loads((ROOT / 'psa_formal' / 'merged' / 'MERGED_MANIFEST.json').read_text(encoding='utf-8'))
formal_qc = pd.read_csv(ROOT / 'psa_formal' / 'merged' / '00_QC.csv')
display(formal_qc)
print(formal_manifest)
assert formal_manifest['formal_psa_run'] is True
assert formal_manifest['draws'] == 10000
assert (formal_qc['status'] == 'PASS').all()

## 4. 上传内容

运行结束后，将整个`psa_formal`文件夹压缩为ZIP并上传。不要只上传图片；正式审计需要`merged`和20个批次目录。